# CPI Prediction — Data Collection

Downloads raw data from all five sources and saves them as CSVs in `data/raw/`.

**Run this notebook first, then run `02_preprocess_and_eda.ipynb`.**

| Source | Output file | Key needed? |
|--------|-------------|-------------|
| FRED | `data/raw/fred_data.csv` | No (public CSV endpoint) |
| Philadelphia Fed SPF | `data/raw/spf_data.csv` | No |
| Kalshi | `data/raw/kalshi_cpi.csv` | Yes — set `KALSHI_API_KEY` in `.env` |
| GDELT + Google Trends | `data/raw/sentiment_data.csv` | No |

Get a Kalshi API key: kalshi.com → Settings → API Access → Create Key.  
If the key is missing the cell saves a placeholder (all NaN) — the rest of the pipeline still works.

In [ ]:
import os, io, re, time, warnings
import subprocess
import numpy as np
import pandas as pd
import requests
from io import StringIO

warnings.filterwarnings('ignore')

# Always run from project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

os.makedirs('data/raw', exist_ok=True)
print('Working directory:', os.getcwd())

## 1. FRED Macro & Survey Data

Uses the public FRED CSV endpoint — no API key required.  
Collects: CPI, Michigan expectations, Cleveland Fed expectations, unemployment, PPI, PCE, WTI oil.

In [ ]:
START, END = '2021-01-01', '2026-04-30'

FRED_SERIES = {
    'cpi_level':           'CPIAUCSL',
    'michigan_inflation':  'MICH',
    'cleveland_inflation': 'EXPINF1YR',
    'unemployment':        'UNRATE',
    'ppi':                 'PPIACO',
    'pce_level':           'PCEPI',
    'oil_wti':             'DCOILWTICO',
}

def fetch_fred(series_id):
    url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}'
    cmd = f'curl -s --max-time 30 -H "User-Agent: Mozilla/5.0" "{url}"'
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=35)
    if result.returncode != 0 or not result.stdout.strip():
        raise RuntimeError(f'curl failed for {series_id}: {result.stderr.strip()}')
    df = pd.read_csv(StringIO(result.stdout))
    df.columns = ['date', 'value']
    df['date']  = pd.to_datetime(df['date'])
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    df = df[(df['date'] >= START) & (df['date'] <= END)].set_index('date')['value']
    return df.resample('ME').last()

print('Fetching FRED series...')
fred_df = pd.DataFrame({name: fetch_fred(sid) for name, sid in FRED_SERIES.items()})
fred_df.index.name = 'date'

# Compute YoY rates here so they're available in the raw file
fred_df['cpi_yoy'] = fred_df['cpi_level'].pct_change(12) * 100
fred_df['pce_yoy'] = fred_df['pce_level'].pct_change(12) * 100

fred_df.to_csv('data/raw/fred_data.csv')
print(f'Saved data/raw/fred_data.csv — {fred_df.shape[0]} rows x {fred_df.shape[1]} cols')
fred_df.tail(3)

## 2. Philadelphia Fed Survey of Professional Forecasters (SPF)

Downloads `median_cpi_level.xlsx` directly. `CPI2` = one-quarter-ahead median CPI forecast.  
Quarterly data is forward-filled to monthly (each quarter's forecast held until the next release).

In [ ]:
SPF_URL = ('https://www.philadelphiafed.org/-/media/frbp/assets/surveys-and-data/'
           'survey-of-professional-forecasters/data-files/files/median_cpi_level.xlsx')

def load_spf():
    r = requests.get(SPF_URL, timeout=30)
    r.raise_for_status()
    xl = pd.read_excel(io.BytesIO(r.content), sheet_name=0)
    xl = xl[['YEAR', 'QUARTER', 'CPI2']].dropna(subset=['CPI2']).copy()
    xl.columns = ['year', 'quarter', 'spf_cpi_forecast']
    def to_me(row):
        return pd.Timestamp(year=int(row.year), month=int(row.quarter)*3, day=1) + pd.offsets.MonthEnd(0)
    xl['date'] = xl.apply(to_me, axis=1)
    xl = xl[xl.date >= '2021-01-01'].set_index('date')[['spf_cpi_forecast']]
    idx = pd.date_range('2021-01-31', '2026-04-30', freq='ME')
    return xl.reindex(idx).ffill().rename_axis('date')

spf_df = load_spf()
spf_df.to_csv('data/raw/spf_data.csv')
print(f'Saved data/raw/spf_data.csv — {spf_df.shape[0]} rows')
spf_df.tail(4)

## 3. Kalshi Prediction Market Data

Requires a free Kalshi API key.  
Get one at: **kalshi.com → Settings → API Access → Create Key**

Set `KALSHI_API_KEY` in a `.env` file in the project root (or assign it directly below).  
If the key is missing, a placeholder CSV (all NaN) is saved — the pipeline still runs.

In [ ]:
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# ── SET YOUR KALSHI API KEY HERE (or put it in .env) ────────────────────
KALSHI_API_KEY = os.getenv('KALSHI_API_KEY', '')
# ────────────────────────────────────────────────────────────────────────

BASE_K = 'https://trading-api.kalshi.com/trade-api/v2'

def kalshi_implied_cpi(markets_for_month):
    tp = []
    for m in markets_for_month:
        title = m.get('title', '') + ' ' + m.get('subtitle', '')
        match = re.search(r'(\d+\.?\d*)\s*%', title)
        if not match:
            continue
        thr  = float(match.group(1))
        res  = m.get('result', '')
        prob = 1.0 if res == 'yes' else (0.0 if res == 'no' else float(m.get('yes_bid', 0.5)))
        tp.append((thr, prob))
    if len(tp) < 2:
        return None
    tp.sort()
    thrs  = [x[0] for x in tp]
    probs = [x[1] for x in tp]
    exp   = sum((probs[i] - probs[i+1]) * (thrs[i] + thrs[i+1]) / 2 for i in range(len(thrs)-1))
    exp  += (1 - probs[0]) * (thrs[0] - 0.25) + probs[-1] * (thrs[-1] + 0.25)
    return round(exp, 3)

if not KALSHI_API_KEY:
    print('KALSHI_API_KEY not set — saving placeholder (all NaN).')
    print('Add KALSHI_API_KEY to .env and re-run this cell to get real data.')
    dates = pd.date_range('2022-01-31', '2026-04-30', freq='ME')
    kalshi_df = pd.DataFrame({'kalshi_implied_cpi': np.nan}, index=dates)
    kalshi_df.index.name = 'date'
else:
    # API key used directly as Bearer token — no login step needed
    kh = {'Authorization': f'Bearer {KALSHI_API_KEY}'}

    series_r   = requests.get(f'{BASE_K}/series', headers=kh, params={'limit': 200})
    series_r.raise_for_status()
    cpi_series = [s for s in series_r.json().get('series', [])
                  if 'CPI' in s.get('ticker', '').upper() or 'inflation' in s.get('title', '').lower()]
    sticker = cpi_series[0]['ticker']
    print(f'Using Kalshi series: {sticker}')

    markets, cursor = [], None
    while True:
        params = {'series_ticker': sticker, 'status': 'settled', 'limit': 200}
        if cursor:
            params['cursor'] = cursor
        resp = requests.get(f'{BASE_K}/markets', headers=kh, params=params)
        resp.raise_for_status()
        data = resp.json()
        markets.extend(data.get('markets', []))
        cursor = data.get('cursor')
        if not cursor:
            break
    print(f'Fetched {len(markets)} settled markets.')

    MONTH_RE = re.compile(
        r'(Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|'
        r'Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?|'
        r'Dec(?:ember)?)\s+(\d{4})', re.IGNORECASE)
    by_month = {}
    for m in markets:
        hit = MONTH_RE.search(m.get('title', ''))
        if hit:
            by_month.setdefault(hit.group(0), []).append(m)

    rows = []
    for ms, ml in sorted(by_month.items()):
        imp = kalshi_implied_cpi(ml)
        if imp is not None:
            rows.append({'date': pd.to_datetime(ms) + pd.offsets.MonthEnd(0), 'kalshi_implied_cpi': imp})
            print(f'  {ms}: {imp}')
    kalshi_df = pd.DataFrame(rows).set_index('date').sort_index()
    kalshi_df = kalshi_df[kalshi_df.index >= '2022-01-31']
    print(f'Kalshi: {len(kalshi_df)} months of data')

kalshi_df.to_csv('data/raw/kalshi_cpi.csv')
print(f'Saved data/raw/kalshi_cpi.csv — {len(kalshi_df)} rows')
kalshi_df.head()

## 4. GDELT News Sentiment + Google Trends

No API keys needed.
- **GDELT**: average monthly article tone for queries mentioning 'inflation CPI' (scale −100 to +100)
- **Google Trends**: monthly US search volume index for 'inflation' (0–100)

GDELT makes ~52 requests with 0.5s sleep between each — takes ~3 minutes.

In [ ]:
# ── GDELT ─────────────────────────────────────────────────────────────────
def gdelt_tone(year, month):
    start = pd.Timestamp(year=year, month=month, day=1)
    end   = start + pd.offsets.MonthEnd(0)
    try:
        r = requests.get('https://api.gdeltproject.org/api/v2/artlist/artlist',
            params={'query': 'inflation CPI', 'mode': 'artlist', 'maxrecords': 250,
                    'format': 'json',
                    'startdatetime': start.strftime('%Y%m%d000000'),
                    'enddatetime':   end.strftime('%Y%m%d235959')}, timeout=30)
        r.raise_for_status()
        arts  = r.json().get('articles', [])
        tones = [float(a['tone']) for a in arts if 'tone' in a]
        return np.mean(tones) if tones else np.nan
    except Exception:
        return np.nan

dates = pd.date_range('2022-01-31', '2026-04-30', freq='ME')
gdelt_rows = []
for d in dates:
    t = gdelt_tone(d.year, d.month)
    gdelt_rows.append({'date': d, 'gdelt_tone': t})
    time.sleep(0.5)
    label = f'{t:.3f}' if not np.isnan(t) else 'no data'
    print(f'  GDELT {d.strftime("%Y-%m")}: {label}')

gdelt_s = pd.DataFrame(gdelt_rows).set_index('date')['gdelt_tone']
print(f'\nGDELT: {gdelt_s.notna().sum()}/{len(gdelt_s)} months with data')

# ── Google Trends ─────────────────────────────────────────────────────────
from pytrends.request import TrendReq

pt = TrendReq(hl='en-US', tz=0, timeout=(10, 25))
pt.build_payload(['inflation'], cat=0, timeframe='2022-01-01 2026-04-01', geo='US')
time.sleep(2)
gt_raw = pt.interest_over_time()

if not gt_raw.empty:
    gt_raw.index = gt_raw.index + pd.offsets.MonthEnd(0)
    google_s = gt_raw['inflation'].rename('google_trends_inflation')
else:
    google_s = pd.Series(dtype=float, name='google_trends_inflation')

print(f'Google Trends: {len(google_s)} monthly observations')

# ── Save combined sentiment file ───────────────────────────────────────────
sentiment_df = pd.concat([gdelt_s, google_s], axis=1)
sentiment_df.columns = ['gdelt_tone', 'google_trends_inflation']
sentiment_df = sentiment_df[sentiment_df.index >= '2022-01-31']
sentiment_df.to_csv('data/raw/sentiment_data.csv')
print(f'Saved data/raw/sentiment_data.csv — {sentiment_df.shape[0]} rows')
sentiment_df.tail(3)

## Done

All raw data collected:
- `data/raw/fred_data.csv` — FRED macro + survey series
- `data/raw/spf_data.csv` — SPF professional forecasts
- `data/raw/kalshi_cpi.csv` — Kalshi implied CPI (NaN if key not set)
- `data/raw/sentiment_data.csv` — GDELT tone + Google Trends

**Next step:** Run `02_preprocess_and_eda.ipynb` to merge the panel and explore the data.